# Understanding hooks on a 1-layer custom trained transformer

In [43]:
import os
import torch

from pathlib import Path 
from transformers import AutoTokenizer
from transformer.config import Config
from transformer.main import DemoTransformer
from typing import List, Tuple

## Constants and device

In [17]:
TRAINED_MODEL_DIR = "../src/transformer/trained_models"
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS")
else:
    device = torch.device("cpu")
    print("MPS device not found. Using CPU instead.")

Using MPS


## Load model, tokenizer and sanity checks

In [20]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')

In [30]:
cfg = Config(n_layers=1)    
model = DemoTransformer(cfg=cfg).to(device)
model_dir = Path(TRAINED_MODEL_DIR)
checkpoint_path = max(model_dir.glob('*.pth'), key = os.path.getctime)
print(f"Loading model at {checkpoint_path}")
model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
model.eval() # We are only performing inference in this script
print(f"Model set in eval() mode for inference")

Loading model at ../src/transformer/trained_models/model_weights_20241212_170753.pth
Model set in eval() mode for inference


In [23]:
for name, param in model.named_parameters():
    print(f"Layer: {name} | Shape: {param.shape} on device {param.device}")

Layer: embed.W_E | Shape: torch.Size([50257, 768]) on device mps:0
Layer: pos_embed.W_pos | Shape: torch.Size([1024, 768]) on device mps:0
Layer: blocks.0.ln1.W | Shape: torch.Size([768]) on device mps:0
Layer: blocks.0.ln1.b | Shape: torch.Size([768]) on device mps:0
Layer: blocks.0.attn.W_Q | Shape: torch.Size([12, 768, 64]) on device mps:0
Layer: blocks.0.attn.b_Q | Shape: torch.Size([12, 64]) on device mps:0
Layer: blocks.0.attn.W_K | Shape: torch.Size([12, 768, 64]) on device mps:0
Layer: blocks.0.attn.b_K | Shape: torch.Size([12, 64]) on device mps:0
Layer: blocks.0.attn.W_V | Shape: torch.Size([12, 768, 64]) on device mps:0
Layer: blocks.0.attn.b_V | Shape: torch.Size([12, 64]) on device mps:0
Layer: blocks.0.attn.W_O | Shape: torch.Size([12, 64, 768]) on device mps:0
Layer: blocks.0.attn.b_O | Shape: torch.Size([768]) on device mps:0
Layer: blocks.0.ln2.W | Shape: torch.Size([768]) on device mps:0
Layer: blocks.0.ln2.b | Shape: torch.Size([768]) on device mps:0
Layer: blocks.0.

## Inference function

In [56]:
def hooked_inference(text: str, seed=None) -> Tuple[str, List[torch.Tensor]]:
    if seed is not None:
        torch.manual_seed(seed)
    
    # Store activations
    mlp_activations = []
    def hook_fn(module, input, output):
        mlp_activations.append(output.detach().clone())
        print(output)
        return output

    hook = model.blocks[0].mlp.register_forward_hook(hook_fn)
    try:
        tokens = tokenizer(text, return_tensors='pt')
        with torch.no_grad():
            output = model(tokens['input_ids'])

        predicted_token_ids = torch.argmax(output, dim=-1)
        predicted_text = tokenizer.decode(predicted_token_ids[0][-1])
                
    finally:
        hook.remove()

    return predicted_text, mlp_activations


def run_inference(text: str, steer: bool = False) -> None:
    tokens = tokenizer(text, return_tensors='pt')
    if steer: 
        hooked_inference()
    else:                 
        with torch.no_grad():
            output = model(tokens['input_ids'])

        predicted_token_ids = torch.argmax(output, dim=-1)
        predicted_text = tokenizer.decode(predicted_token_ids[0][-1])
    
    return predicted_text



In [57]:

text = "The cat in the"
hooked_inference(text)

tensor([[[10.5516, -1.7158, -7.7320,  ..., -2.1494,  4.9147,  8.7614],
         [ 3.7137, -2.8516, -9.7144,  ..., -2.2397,  0.2513,  3.1898],
         [ 4.1938, -3.0583, -2.3143,  ..., -1.4551,  6.6294, -2.5762],
         [ 2.5142, -1.4812, -1.8916,  ..., -0.2204,  1.6994, -0.5753]]],
       device='mps:0')


(' hat',
 [tensor([[[10.5516, -1.7158, -7.7320,  ..., -2.1494,  4.9147,  8.7614],
           [ 3.7137, -2.8516, -9.7144,  ..., -2.2397,  0.2513,  3.1898],
           [ 4.1938, -3.0583, -2.3143,  ..., -1.4551,  6.6294, -2.5762],
           [ 2.5142, -1.4812, -1.8916,  ..., -0.2204,  1.6994, -0.5753]]],
         device='mps:0')])